# Phase 5 — Robustness & Motif Analysis
**Steps 5.1 – 5.6** | Attack simulation, Superpower Fragility Index, frustrated triangle motifs, C(k) hierarchy.

**Deliverable D3:** Fragility Index table + attack/removal curves.

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats as scipy_stats
from itertools import combinations
import matplotlib.pyplot as plt
from tqdm import tqdm
import community as community_louvain
warnings.filterwarnings('ignore')

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150,
                    'grid.color': '#30363d', 'grid.alpha': 0.5}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')
PROC   = os.path.join(ROOT, 'data', 'processed')

def load_graph(name):
    with open(os.path.join(NETS, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

ERAS = ['full', 'cold_war', 'post_cw', 'post_9_11', 'recent']
G_full = load_graph('full')
print(f'Full network: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges')

## Steps 5.1 + 5.2 – Targeted vs Random Attack Simulation

In [ ]:
def attack_simulation(G, removal_order, stop_frac=0.5):
    """Simulate iterative node removal. Record S, <d>, and Q."""
    H = G.copy()
    N_orig = H.number_of_nodes()
    max_remove = int(stop_frac * N_orig)
    results = []

    for i, node in enumerate(removal_order):
        if i >= max_remove: break
        if node not in H: continue
        H.remove_node(node)
        
        if H.number_of_nodes() == 0:
            results.append({'removed_frac': (i+1)/N_orig, 'giant_frac': 0, 'avg_path': np.nan, 'modularity_Q': np.nan})
            break
            
        gcc = max(nx.connected_components(H), key=len)
        G_gcc = H.subgraph(gcc).copy()
        giant_frac = len(gcc) / N_orig

        # Average path length (sampled for speed)
        sample_size = min(30, G_gcc.number_of_nodes())
        if G_gcc.number_of_edges() > 0 and sample_size > 1:
            sample = np.random.choice(list(G_gcc.nodes), size=sample_size, replace=False)
            lengths = []
            for s in sample:
                spl = nx.single_source_shortest_path_length(G_gcc, s)
                lengths.extend(spl.values())
            avg_path = np.mean(lengths)
        else:
            avg_path = np.nan

        # Modularity
        try:
            partition_tmp = community_louvain.best_partition(H, weight='weight', random_state=0)
            Q_tmp = community_louvain.modularity(partition_tmp, H, weight='weight')
        except Exception:
            Q_tmp = np.nan

        results.append({
            'removed_frac': (i+1)/N_orig,
            'giant_frac':   giant_frac,
            'avg_path':     avg_path,
            'modularity_Q': Q_tmp
        })
    return pd.DataFrame(results)

In [ ]:
# Step 5.2 - Execution (Targeted and Random)
# 1. Targeted Attack: Betweenness Centrality
print('Calculating targeted removal order (betweenness)...')
betweenness = nx.betweenness_centrality(G_full, weight='weight')
targeted_order = sorted(G_full.nodes, key=lambda n: betweenness[n], reverse=True)
targeted_df = attack_simulation(G_full, targeted_order)

# 2. Random Failure: Sample 10 runs
print('Calculating random failures (10 runs)...')
n_rand_runs = 10
x_grid = targeted_df['removed_frac'].values
rand_results = []
for i in range(n_rand_runs):
    rand_order = list(G_full.nodes)
    np.random.shuffle(rand_order)
    res = attack_simulation(G_full, rand_order)
    rand_results.append(res['giant_frac'].values)

rand_matrix = np.array(rand_results)
rand_mean = np.mean(rand_matrix, axis=0)
rand_std  = np.std(rand_matrix, axis=0)
print('Execution complete.')

In [ ]:
# Plot targeted vs random for full network
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Network Robustness: Full UNGA Voting Network', color='white', fontsize=13)

ax.plot(targeted_df['removed_frac'], targeted_df['giant_frac'],
        'o-', color='#f85149', lw=2, ms=4, label='Targeted attack (betweenness)')
ax.plot(x_grid, rand_mean, '-', color='#3fb950', lw=2, label='Random failure (mean)')
ax.fill_between(x_grid, rand_mean - rand_std, rand_mean + rand_std,
                alpha=0.25, color='#3fb950', label='±1 SD random')

ax.set_xlabel('Fraction of nodes removed')
ax.set_ylabel('Giant component (fraction of original N)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_robustness_full.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
def run_robustness(G, n_random_runs=10, stop_frac=0.5):
    """Wrapper to calculate targeted (betweenness) and random failure curves."""
    # 1. Targeted Attack
    betweenness = nx.betweenness_centrality(G, weight='weight')
    targeted_order = sorted(G.nodes, key=lambda n: betweenness[n], reverse=True)
    t_df = attack_simulation(G, targeted_order, stop_frac=stop_frac)
    
    # 2. Random Failure
    x_grid = t_df['removed_frac'].values
    rand_results = []
    for i in range(n_random_runs):
        rand_order = list(G.nodes)
        np.random.shuffle(rand_order)
        res = attack_simulation(G, rand_order, stop_frac=stop_frac)
        rand_results.append(res['giant_frac'].values)
    
    rand_matrix = np.array(rand_results)
    rand_mean = np.mean(rand_matrix, axis=0)
    rand_std  = np.std(rand_matrix, axis=0)
    
    return t_df, x_grid, rand_mean, rand_std, targeted_order

## Step 5.3 – Robustness on 4 Temporal Sub-networks

In [ ]:
temporal_eras = ['cold_war', 'post_cw', 'post_9_11', 'recent']
era_labels = {'cold_war': 'Cold War (1946–91)', 'post_cw': 'Post-CW (1991–01)',
              'post_9_11': 'Post-9/11 (2001–14)', 'recent': 'Recent (2014–15)'}
era_colors = ['#58a6ff', '#f85149', '#3fb950', '#d2a8ff']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Network Robustness Across Eras', color='white', fontsize=14)
axes = axes.flatten()

era_robustness = {}
for idx, era in enumerate(temporal_eras):
    G_era = load_graph(era)
    if G_era.number_of_edges() < 5:
        print(f'{era}: too few edges, skipping')
        continue
    print(f'\nRunning robustness for {era}...')
    t_df, x_g, r_mean, r_std, t_order = run_robustness(G_era, n_random_runs=30, stop_frac=0.5)
    era_robustness[era] = {'targeted': t_df, 'rand_x': x_g,
                           'rand_mean': r_mean, 'targeted_order': t_order}

    ax = axes[idx]
    ax.plot(t_df['removed_frac'], t_df['giant_frac'],
            'o-', color='#f85149', lw=2, ms=3, label='Targeted')
    ax.plot(x_g, r_mean, '-', color='#3fb950', lw=2, label='Random (mean)')
    ax.fill_between(x_g, r_mean-r_std, r_mean+r_std, alpha=0.2, color='#3fb950')
    ax.set_title(era_labels[era], color='white')
    ax.set_xlabel('Fraction removed'); ax.set_ylabel('Giant component')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 0.5); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_robustness_eras.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 5.4 – Superpower Fragility Index (D3)

In [ ]:
from numpy import trapz
# Area under the targeted-removal curve (lower area = more fragile)
frag_rows = []
for era, data in era_robustness.items():
    t_df = data['targeted']
    auc = trapz(t_df['giant_frac'], t_df['removed_frac'])
    frag_rows.append({'era': era, 'targeted_AUC': round(auc, 5)})
print('Era fragility (AUC):', frag_rows)

# Per-country fragility: rank * slope
country_frag = []
# Assuming targeted_order and targeted_df from Step 5.2 are available
for rank, country in enumerate(targeted_order[:20]):
    if rank == 0: s_before = 1.0
    else: s_before = targeted_df.iloc[rank-1]['giant_frac']
    s_after = targeted_df.iloc[rank]['giant_frac'] if rank < len(targeted_df) else 0
    delta_S = s_before - s_after
    country_frag.append({'country': country, 'rank': rank+1, 'delta_S': round(delta_S, 5)})
frag_df = pd.DataFrame(country_frag)
print(frag_df.head(10))

In [ ]:
# Plot fragility index
top10 = frag_df.head(10)
name_map = {'United States of America': 'USA', 'South Africa': 'S.Africa'}
top10_names = [name_map.get(c, c) for c in top10['country']]

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Superpower Fragility Index — Top 10 (D3)', color='white', fontsize=13)
colors = plt.cm.YlOrRd(np.linspace(0.9, 0.3, 10))
bars = ax.barh(top10_names[::-1], top10['delta_S'].values[::-1], color=colors)
ax.set_xlabel('ΔS (drop in giant component when removed first)')
ax.set_title('Diplomatic Structural Indispensability', color='white')

for bar, val in zip(bars, top10['delta_S'].values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8, color='white')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_fragility_index.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 5.5 – Frustrated Triangle Motif Search

In [ ]:
# Proper signed balance analysis
agree_matrix = pd.read_csv(os.path.join(PROC, 'agree_matrix_full.csv'), index_col=0)
G_signed = nx.Graph()
countries = agree_matrix.columns.tolist()
for i in range(len(countries)):
    for j in range(i+1, len(countries)):
        w = agree_matrix.iloc[i, j]
        if not np.isnan(w):
            G_signed.add_edge(countries[i], countries[j],
                              weight=w,
                              sign=1 if w >= 0.5 else -1)

n_balanced = 0
n_unbalanced = 0
np.random.seed(42)
sample_nodes = np.random.choice(list(G_signed.nodes), size=min(80, len(G_signed.nodes)), replace=False).tolist()
for a, b, c in combinations(sample_nodes, 3):
    if G_signed.has_edge(a,b) and G_signed.has_edge(b,c) and G_signed.has_edge(a,c):
        product = (G_signed[a][b]['sign'] *
                   G_signed[b][c]['sign'] *
                   G_signed[a][c]['sign'])
        if product == 1: n_balanced += 1
        else: n_unbalanced += 1

balance_ratio = n_balanced / (n_balanced + n_unbalanced + 1e-9)
print(f'Structural balance ratio: {balance_ratio:.4f}')
print(f'(1.0 = perfectly balanced, 0.5 = random, <0.5 = frustrated)')

## Step 5.6 – C(k) Hierarchical Model Check (Per Era)

In [ ]:
from scipy import stats as scipy_stats

hier_results = []
fig, ax = plt.subplots(figsize=(9, 6))
fig.suptitle('C(k) Hierarchical Check — All Eras (log-log)', color='white', fontsize=13)

colors_map = {'full':'#58a6ff','cold_war':'#f85149','post_cw':'#3fb950',
              'post_9_11':'#d2a8ff','recent':'#ffa657'}

for era in ERAS:
    G = load_graph(era)
    c_dict = nx.clustering(G)
    degs   = np.array([G.degree(n) for n in G.nodes])
    clusts = np.array([c_dict[n] for n in G.nodes])

    # Bin
    df_ck = pd.DataFrame({'k': degs, 'c': clusts})
    ck = df_ck[df_ck['k'] >= 2].groupby('k')['c'].mean()

    log_k = np.log10(np.clip(ck.index.astype(float).values, 1, None))
    log_c = np.log10(ck.values.clip(min=1e-6))
    valid  = np.isfinite(log_k) & np.isfinite(log_c)

    slope = np.nan
    if valid.sum() >= 3:
        slope, intercept, r, p, _ = scipy_stats.linregress(log_k[valid], log_c[valid])
        x_fit = np.linspace(log_k[valid].min(), log_k[valid].max(), 50)
        ax.plot(x_fit, intercept + slope*x_fit, '--', color=colors_map[era], lw=1.5)

    ax.scatter(log_k, log_c, s=20, alpha=0.5, color=colors_map[era],
               label=f'{era} (β={slope:.2f})' if not np.isnan(slope) else era)
    hier_results.append({'era': era, 'ck_slope': round(slope, 3) if not np.isnan(slope) else np.nan})

# Reference line slope=-1
x_ref = np.linspace(0.3, 2.2, 50)
ax.plot(x_ref, -x_ref + 0.2, 'w--', lw=1, alpha=0.3, label='slope=−1 (hierarchical)')
ax.set_xlabel('log₁₀(k)'); ax.set_ylabel('log₁₀ C(k)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p5_ck_hierarchy_eras.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

hier_df = pd.DataFrame(hier_results)
hier_df.to_csv(os.path.join(TABLES, 'p5_hierarchy_slopes.csv'), index=False)
print(hier_df.to_string(index=False))

## ✅ Phase 5 Complete
**Deliverable D3:**
- `results/tables/p5_fragility_index_D3.csv` — Superpower Fragility Index
- `results/plots/p5_robustness_full.png` — attack/random curves, full network
- `results/plots/p5_robustness_eras.png` — attack curves per era
- `results/plots/p5_fragility_index.png` — top-10 fragility bar chart
- `results/tables/p5_frustrated_motifs.csv` — Z-score for frustrated triangles
- `results/tables/p5_hierarchy_slopes.csv` — C(k) slopes per era

**→ Proceed to Notebook 06: Dynamics & Synthesis**